In [2]:
import os
import mysql.connector
from dotenv import load_dotenv

load_dotenv()

def get_connection():
    return mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        ssl_ca=os.getenv("DB_SSL_CA")
    )

# Prueba
conn = get_connection()
conn.close()

In [3]:
import pandas as pd
import google.genai as genai
from tabulate import tabulate
from IPython.display import display


In [4]:
from google import genai

client = genai.Client(api_key=os.getenv("LLM_API_KEY"))

In [5]:
SYSTEM_PROMPT = """
Sos un experto en bases de datos MySQL. Tu única tarea es convertir preguntas 
en español a consultas SQL válidas para la base de datos 'biblioia'.

REGLAS:
- Respondé ÚNICAMENTE con la consulta SQL, sin explicaciones ni comentarios.
- No uses bloques de código ni backticks.
- Preferí usar las VISTAS disponibles cuando corresponda.
- Si la pregunta no se puede responder con el esquema dado, respondé: 
  SELECT 'No puedo responder esa pregunta con los datos disponibles';

=== ESQUEMA ===

GENERO (id_genero INT PK, nombre VARCHAR(60) UNIQUE NOT NULL, descripcion VARCHAR(255))
AUTOR (id_autor INT PK, nombre VARCHAR(80) NOT NULL, apellido VARCHAR(80) NOT NULL, nacionalidad VARCHAR(60))
LIBRO (isbn VARCHAR(20) PK, titulo VARCHAR(200) NOT NULL, anio_publicacion YEAR, stock_total SMALLINT, stock_disponible SMALLINT)
  -- stock_disponible <= stock_total siempre
LIBRO_AUTOR (isbn FK->LIBRO, id_autor FK->AUTOR) -- N:M
LIBRO_GENERO (isbn FK->LIBRO, id_genero FK->GENERO) -- N:M
SOCIO (id_socio INT PK, dni VARCHAR(15) UNIQUE, nombre VARCHAR(80), apellido VARCHAR(80), email VARCHAR(120) UNIQUE, fecha_alta DATE, estado VARCHAR(12))
  -- estado: 'ACTIVO', 'SUSPENDIDO', 'BAJA'
EJEMPLAR (id_ejemplar INT PK, isbn FK->LIBRO, nro_ejemplar SMALLINT, estado_fisico VARCHAR(12))
  -- estado_fisico: 'BUENO', 'DETERIORADO', 'BAJA'
PRESTAMO (id_prestamo INT PK, id_socio FK->SOCIO, id_ejemplar FK->EJEMPLAR, fecha_prestamo DATE, fecha_vencimiento DATE, fecha_devolucion DATE NULL, estado VARCHAR(12))
  -- estado: 'ACTIVO', 'DEVUELTO', 'VENCIDO'
SANCION (id_sancion INT PK, id_socio FK->SOCIO, tipo VARCHAR(20), fecha_inicio DATE, fecha_fin DATE, motivo VARCHAR(255))
  -- tipo: 'MORA', 'DAÑO', 'PERDIDA', 'OTRO'
  -- activa cuando fecha_fin >= CURRENT_DATE
AUDITORIA_PRESTAMOS (id_audit INT PK, id_prestamo INT, operacion VARCHAR(10), estado_nuevo VARCHAR(12), estado_viejo VARCHAR(12), usuario_bd VARCHAR(80), fecha_hora DATETIME)

=== VISTAS DISPONIBLES ===

v_prestamos_vencidos (id_prestamo, dni, socio, titulo, fecha_prestamo, fecha_vencimiento, dias_de_mora)
  -- préstamos con estado='VENCIDO' y sin devolución

v_prestamos_activos (id_prestamo, id_socio, dni, socio, email, isbn, titulo, nro_ejemplar, fecha_prestamo, fecha_vencimiento, dias_vencido)
  -- todos los préstamos con estado='ACTIVO'

v_libros_disponibles (isbn, titulo, stock_disponible, generos, autores)
  -- libros con stock_disponible > 0 y al menos un ejemplar en buen estado

v_historial_socios (id_socio, dni, socio, isbn, titulo, fecha_prestamo, fecha_vencimiento, fecha_devolucion, estado_prestamo)
  -- historial completo de préstamos por socio

v_libros_mas_prestados (isbn, titulo, autores, total_prestamos)
  -- ranking de libros por cantidad de préstamos

v_socios_sancionados (id_socio, dni, socio, estado, tipo, fecha_inicio, fecha_fin, motivo, dias_restantes)
  -- socios con sanciones activas hoy

v_autores_prolíficos (id_autor, autor, nacionalidad, cantidad_libros)
  -- autores con más de 1 libro en la biblioteca

=== EJEMPLOS ===

Pregunta: ¿Cuáles son los 5 libros más prestados este año?
SQL: SELECT isbn, titulo, total_prestamos FROM v_libros_mas_prestados LIMIT 5;

Pregunta: ¿Qué socios tienen préstamos vencidos en este momento?
SQL: SELECT DISTINCT dni, socio FROM v_prestamos_vencidos;

Pregunta: ¿Qué libros de ciencia ficción están disponibles para prestar?
SQL: SELECT isbn, titulo, stock_disponible FROM v_libros_disponibles WHERE generos LIKE '%Ciencia Ficción%';
"""

In [6]:
def text_to_sql(pregunta: str) -> str:
    respuesta = client.models.generate_content(
       model="gemini-2.5-flash",
        contents=SYSTEM_PROMPT + f"\nPregunta: {pregunta}\nSQL:"
    )
    sql = respuesta.text.strip()
    # Por las dudas, limpiamos backticks que Gemini a veces agrega
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


In [7]:
def ejecutar_consulta(sql: str) -> pd.DataFrame:
    conn = get_connection()
    try:
        df = pd.read_sql(sql, conn)
        return df
    except Exception as e:
        return pd.DataFrame({"Error": [str(e)]})
    # cirra la conexion 
    finally:
        conn.close()

In [8]:
def agente_responder(pregunta: str, mostrar_sql: bool = True):
    print(f"\n Pregunta: {pregunta}")
    print("-" * 60)
    
    sql = text_to_sql(pregunta)
    
    if mostrar_sql:
        print(f" SQL generado:\n{sql}")
        print("-" * 60)
    
    df = ejecutar_consulta(sql)
    
    if df.empty:
        print(" La consulta no devolvió resultados.")
    else:
        print(f" Resultado ({len(df)} filas):")
        display(df)
    
    return df

In [13]:
def obtener_perfil_socio(id_socio: int) -> dict:
    conn = get_connection()
    try:
        # Géneros que leyó
        df_generos = pd.read_sql(
            "SELECT genero FROM v_generos_por_socio WHERE id_socio = %s",
            conn, params=(id_socio,)
        )
        # Autores que leyó
        df_autores = pd.read_sql(
            "SELECT autor FROM v_autores_por_socio WHERE id_socio = %s",
            conn, params=(id_socio,)
        )
        return {
            "generos": df_generos['genero'].tolist(),
            "autores": df_autores['autor'].tolist()
        }
    finally:
        conn.close()

In [14]:
def obtener_libros_candidatos(id_socio: int) -> pd.DataFrame:
    conn = get_connection()
    try:
        df = pd.read_sql(
            "SELECT isbn, titulo, stock_disponible FROM v_libros_no_leidos_por_socio WHERE id_socio = %s",
            conn, params=(id_socio,)
        )
        return df
    finally:
        conn.close()

In [ ]:
def recomendar_para(id_socio: int):
    print(f"\n Recomendaciones para el socio ID {id_socio}")
    print("-" * 60)
    
    perfil = obtener_perfil_socio(id_socio)
    
    if not perfil['generos']:
        print("Este socio no tiene historial de préstamos.")
        return
    
    print(f" Géneros favoritos: {', '.join(perfil['generos'])}")
    print(f" Autores leídos: {', '.join(perfil['autores'])}")
    
    df_candidatos = obtener_libros_candidatos(id_socio)
    
    if df_candidatos.empty:
        print(" No hay libros disponibles para recomendar.")
        return

    libros_str = df_candidatos[['titulo']].to_string(index=False)
    
    prompt = f"""
Sos un bibliotecario amigable . Un socio tiene estos gustos:
- Géneros favoritos: {', '.join(perfil['generos'])}
- Autores que ya leyó: {', '.join(perfil['autores'])}

Estos libros están disponibles y el socio aún no los leyó:
{libros_str}

Recomendá los 3 mejores. 
Para cada uno indicá el título y explicá en 2-3 oraciones por qué le va a gustar.
"""
    
    respuesta = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    print("\n Recomendaciones:")
    print("=" * 60)
    print(respuesta.text)


 Recomendaciones para el socio ID 1
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_12684\896487870.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_generos = pd.read_sql(
C:\Users\miran\AppData\Local\Temp\ipykernel_12684\896487870.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_autores = pd.read_sql(


 Géneros favoritos: Ciencia Ficción
 Autores leídos: Isaac Asimov


C:\Users\miran\AppData\Local\Temp\ipykernel_12684\1794459765.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(



 Recomendaciones:
¡Hola! ¡Qué gusto verte por aquí! Con esos gustos tan definidos, sé exactamente por dónde empezar. Si te encanta la ciencia ficción y ya devoraste a Isaac Asimov, te tengo unas recomendaciones que te van a volar la cabeza.

Aquí te van mis 3 elegidos, pensando en ese amor por la exploración de lo desconocido, la inteligencia artificial y las grandes preguntas sobre la humanidad:

1.  **2001: Una odisea del espacio**
    Este es un clásico absoluto de Arthur C. Clarke, un autor que comparte con Asimov la maestría para la ciencia ficción dura y la especulación científica. Te va a encantar por su ambición cósmica, explorando la evolución humana, la inteligencia artificial y el primer contacto con lo desconocido a una escala monumental. Prepara tu mente para un viaje lleno de asombro y misterio.

2.  **Cita con Rama**
    Otra joya de Arthur C. Clarke, que seguro resonará con tu aprecio por las narrativas de Asimov. Esta novela te sumerge en una fascinante historia de pr

In [20]:
recomendar_para("socio con ID 1")


 Recomendaciones para el socio ID socio con ID 1
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_12684\896487870.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_generos = pd.read_sql(
C:\Users\miran\AppData\Local\Temp\ipykernel_12684\896487870.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_autores = pd.read_sql(


Este socio no tiene historial de préstamos.
